In [ ]:
sepsis_readm=pd.read_csv("sepsis_readm.csv")

In [ ]:
# join with drgcode

drg_group=duckdb.sql("""
SELECT a.*,b.drg_type,b.drg_code,
FROM sepsis_readm a
JOIN 'drgcodes.csv' b ON a.hadm_id=b.hadm_id
;""").df()

print(drg_group)

In [ ]:
inv_pcd=duckdb.sql("""
SELECT 
    hadm_id,
    
    -- 1. Mechanical Ventilation Flag (ICD-9 & ICD-10)
    MAX(
        CASE 
            WHEN icd_version = 10 AND icd_code LIKE '5A19%' THEN 1 
            WHEN icd_version = 9 AND icd_code IN ('9670', '9671', '9672') THEN 1
            ELSE 0 
        END
    ) AS has_mech_vent,

    -- 2. Dialysis / Renal Replacement Flag (ICD-9 & ICD-10)
    MAX(
        CASE 
            WHEN icd_version = 10 AND icd_code LIKE '5A1D%' THEN 1 
            WHEN icd_version = 9 AND icd_code IN ('3995', '5498') THEN 1
            ELSE 0 
        END
    ) AS has_dialysis,

    -- 3. Composite High-Cost Procedure Flag (Any major ICU intervention)
    MAX(
        CASE 
            WHEN (icd_version = 10 AND (icd_code LIKE '5A19%' OR icd_code LIKE '5A1D%' OR icd_code LIKE '5A15%'))
              OR (icd_version = 9 AND icd_code IN ('9670', '9671', '9672', '3995', '3965')) THEN 1
            ELSE 0 
        END
    ) AS has_high_cost_procedure

FROM 'procedures_icd.csv'
GROUP BY hadm_id;
""").df()

print(inv_pcd)

In [ ]:
# join with procedures
sepsis_drg_pcd=duckdb.sql("""
SELECT 
    a.*,
    b.*,
FROM drg_group a
JOIN inv_pcd b 
    ON a.hadm_id=b.hadm_id
;""").df()

print(sepsis_drg_pcd)

In [ ]:
sepsis_readm= sepsis_drg_pcd.query("sepsis_cohort == 1 and drg_type== 'HCFA' ")

print(sepsis_readm)

In [ ]:
# pivot for better visualization
sepsis_readm_pivot=duckdb.sql("""
SELECT 
    hadm_id, 
    readmitted_30d,
    'Mechanical Ventilation' AS procedure_name, 
    has_mech_vent AS procedure_flag
FROM sepsis_readm
UNION ALL
SELECT 
    hadm_id,
    readmitted_30d, 
    'Renal Dialysis' AS procedure_name, 
    has_dialysis AS procedure_flag
FROM sepsis_readm
UNION ALL
SELECT 
    hadm_id,
    readmitted_30d, 
    'High Cost' AS procedure_name, 
    has_high_cost_procedure AS procedure_flag
FROM sepsis_readm;
;""").df()
print(sepsis_readm_pivot)

In [ ]:
sepsis_readm_severity=duckdb.sql("""
SELECT a.*,b.drg_severity,
FROM sepsis_readm a
JOIN 'drgcodes.csv' b ON a.hadm_id=b.hadm_id
WHERE b.drg_severity IS NOT NULL
;""").df()

print(sepsis_readm_severity)

In [ ]:
sepsis_readm.to_csv("/Users/xumuyao/Desktop/JOBHUNTING/mockdata/hosp/sepsis_cohort_analysis.csv")
sepsis_readm_pivot.to_csv("/Users/xumuyao/Desktop/JOBHUNTING/mockdata/hosp/sepsis_cohort_long.csv")
sepsis_readm_severity.to_csv("/Users/xumuyao/Desktop/JOBHUNTING/mockdata/hosp/sepsis_cohort_severity.csv")